In [1]:
!pip install pandas


[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [9]:
import pandas
import re

year = 2026

In [10]:
data = pandas.read_csv(f"data/{year}/{year}.csv")
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 27133 entries, 0 to 27132
Data columns (total 25 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   _id                     27133 non-null  int64  
 1   DATA                    27133 non-null  str    
 2   HORA                    27133 non-null  str    
 3   CONCESSIONARIA          27133 non-null  str    
 4   RODOVIA                 27133 non-null  str    
 5   KM                      27133 non-null  float64
 6   SENTIDO                 27133 non-null  str    
 7   LATITUDE                27130 non-null  float64
 8   LONGITUDE               27130 non-null  float64
 9   CLASSE                  27133 non-null  str    
 10  SUBCLASSE               27133 non-null  str    
 11  CAUSA_PROVAVEL          27133 non-null  str    
 12  VITIMA_ILESA            27133 non-null  int64  
 13  VITIMA_LEVE             27133 non-null  int64  
 14  VITIMA_MODERADA         27133 non-null  int64  
 

In [11]:
# Safely convert string to dictionary
def safe_eval(cell: str):
    try:
        pairs = {}

        vehicles = cell.split("|")
        for vech in vehicles:
            partial = vech.split("=")
            pairs[partial[0]] = int(partial[1])
        return pairs
    except:
        return {}

# Apply conversion
df_dicts = data.VEICULOS_ENVOLVIDOS.apply(safe_eval)

df_dicts.info()

# Expand dictionaries into separate columns
df_expanded = pandas.json_normalize(df_dicts).fillna(0)

data = pandas.concat([data, df_expanded], axis=1)
data.drop(columns="VEICULOS_ENVOLVIDOS")



<class 'pandas.Series'>
RangeIndex: 27133 entries, 0 to 27132
Series name: VEICULOS_ENVOLVIDOS
Non-Null Count  Dtype 
--------------  ----- 
27133 non-null  object
dtypes: object(1)
memory usage: 212.1+ KB


,_id,DATA,HORA,CONCESSIONARIA,RODOVIA,KM,SENTIDO,LATITUDE,LONGITUDE,CLASSE,...,ÔNIBUS,NÃO SE APLICA,MICRO-ÔNIBUS,TRATOR,UTILITÁRIO,CARRETINHA,TRAILER,CARROÇA,CICLOMOTOR,CHARRETE
0,1,2026-01-01T00:00:00,05:13:08,L01-AUTOBAN,SP330,72.900,NORTE,-23.083270,-46.977380,COLISÃO,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2,2026-01-01T00:00:00,06:34:01,L01-AUTOBAN,SP348,60.000,NORTE,-23.176079,-46.936076,CHOQUE,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,3,2026-01-01T00:00:00,11:50:33,L01-AUTOBAN,SP348,105.000,NORTE,-22.898464,-47.214528,COLISÃO,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,4,2026-01-01T00:00:00,22:04:16,L01-AUTOBAN,SP330,63.000,SUL,-23.160879,-46.931920,COLISÃO,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,5,2026-01-02T00:00:00,04:24:53,L01-AUTOBAN,SP348,53.800,NORTE,-23.222733,-46.907420,ATROPELAMENTO DE ANIMAL,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
27128,27129,2026-07-06T00:00:00,17:31:07,L34-RAPOSO CASTELLO,SPM280D,25.000,OESTE,-23.504389,-46.863911,COLISÃO,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
27129,27130,2026-07-27T00:00:00,08:13:11,L34-RAPOSO CASTELLO,SPM280E,24.000,LESTE,-23.505676,-46.854362,COLISÃO,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
27130,27131,2026-08-04T00:00:00,20:13:32,L34-RAPOSO CASTELLO,SPM280D,25.000,OESTE,-23.504389,-46.863911,CHOQUE,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
27131,27132,2026-08-08T00:00:00,13:43:22,L34-RAPOSO CASTELLO,SPM280D,26.000,OESTE,-23.504623,-46.872849,COLISÃO,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [12]:
def normalizar_classe(s):
    if pandas.isna(s):
        return "NÃO INFORMADO"
    s = s.upper().strip()
    if re.search(r"OBJETO LAN.ADO", s):
        return "OBJETO LANÇADO CONTRA O VEÍCULO"
    return s

data["CLASSE"] = data["CLASSE"].apply(normalizar_classe)

In [13]:
def normalizar_subclasse(s):
    if pandas.isna(s):
        return "NÃO INFORMADO"
    s = s.upper().strip()
    
    regras = [
        (r"DEFENSA|BARREIRA|SUBMARINO", "CHOQUE-DEFENSA/BARREIRA"),
        (r"DRENAGEM", "CHOQUE-ELEMENTO DE DRENAGEM"),
        (r"TALUDE|BARRANCO|CORTE", "CHOQUE-TALUDE/BARRANCO"),
        (r"MEIO.?FIO|CALÇAMENTO", "CHOQUE-MEIO FIO"),
        (r"ÁRVORE|ARVORE", "CHOQUE-ÁRVORE"),
        (r"POSTE", "CHOQUE-POSTE"),
        (r"BURACO", "CHOQUE-BURACO"),
        (r"OBJETO.*PISTA|OBJETO SOBRE A VIA|VEÍCULO PARADO NA PISTA", "CHOQUE-OBJETO NA PISTA"),
        (r"VEÍCULO PARADO NO ACOSTAMENTO", "CHOQUE-VEÍCULO PARADO NO ACOSTAMENTO"),
        (r"OAE|PILAR|VIADUTO|PONTE", "CHOQUE-OAE (PONTE/VIADUTO)"),
        (r"PRAÇA|CABINE|CANCELA|PEDÁGIO", "CHOQUE-PRAÇA DE PEDÁGIO"),
        (r"SINALIZAÇÃO|EQUIPAMENTO|PAINEL", "CHOQUE-SINALIZAÇÃO/EQUIPAMENTO"),
        (r"CERCA|ALAMBRADO|MOURÃO", "CHOQUE-CERCAS/ALAMBRADOS"),
        (r"EDIFICAÇÃO|ILHA|MATACÃO|OUTROS|NÃO IDENTIF", "CHOQUE-OUTROS"),
        
        (r"^FRONTAL$|COLIS.O-FRONTAL", "COLISÃO-FRONTAL"),
        (r"^TRASEIRA$|COLIS.O-TRASEIRA", "COLISÃO-TRASEIRA"),
        (r"^LATERAL$|COLIS.O-LATERAL", "COLISÃO-LATERAL"),
        (r"^TRANSVERSAL$|COLIS.O-TRANSVERSAL", "COLISÃO-TRANSVERSAL"),
        
        (r"^TOMBAMENTO$", "TOMBAMENTO"),
        (r"TOMBAMENTO-MOTO", "TOMBAMENTO-MOTO"),
        (r"TOMBAMENTO.*PESAD", "TOMBAMENTO-VEÍCULO PESADO"),
        (r"TOMBAMENTO-BICICLETA", "TOMBAMENTO-BICICLETA"),
        
        (r"^CAPOTAMENTO$", "CAPOTAMENTO"),
        (r"^ENGAVETAMENTO$", "ENGAVETAMENTO"),
        
        (r"SUICID|SUICÍD", "ATROP. PEDESTRE-SUICÍDIO"),
        (r"CICLISTA", "ATROP. PEDESTRE-CICLISTA"),
        (r"ATROP\.? PEDESTRE|DE PEDESTRE|PEDESTRE USUÁRIO", "ATROP. PEDESTRE-OUTROS"),
        
        (r"ANIMAL.*SILVESTRE.*GRANDE", "ATROP. ANIMAL-SILVESTRE GRANDE"),
        (r"ANIMAL.*SILVESTRE.*M.DIO", "ATROP. ANIMAL-SILVESTRE MÉDIO"),
        (r"ANIMAL.*SILVESTRE.*PEQUENO", "ATROP. ANIMAL-SILVESTRE PEQUENO"),
        (r"ANIMAL.*DOM.STICO.*GRANDE", "ATROP. ANIMAL-DOMÉSTICO GRANDE"),
        (r"ANIMAL.*DOM.STICO.*M.DIO", "ATROP. ANIMAL-DOMÉSTICO MÉDIO"),
        (r"ANIMAL.*DOM.STICO.*PEQUENO", "ATROP. ANIMAL-DOMÉSTICO PEQUENO"),
        (r"ANIMAL", "ATROP. ANIMAL-OUTROS"),
        
        (r"^QUEDA-MOTO$", "QUEDA-MOTO"),
        (r"^QUEDA-CICLISTA$", "QUEDA-CICLISTA"),
        (r"RIBANCEIRA|EM RIBANCEIRA", "QUEDA-RIBANCEIRA/OAE"),
        (r"QUEDA-CARGA", "QUEDA-CARGA"),
        (r"^QUEDA$|TABLUDE", "QUEDA-OUTROS"),
        
        (r"LAN.ADO", "OBJETO LANÇADO CONTRA O VEÍCULO"),
        (r"INC.NDIO", "INCÊNDIO"),
        (r"SA.DA DE PISTA", "SAÍDA DE PISTA"),
    ]
    
    for padrao, categoria in regras:
        if re.search(padrao, s):
            return categoria
    
    return "OUTROS/NÃO CLASSIFICADO"

data["SUBCLASSE"] = data["SUBCLASSE"].apply(normalizar_subclasse)

In [20]:
data = data.sort_values(by=[data.DATA.name, data.HORA.name], ascending=True)

data.loc[(data.RODOVIA.str.contains("330")), :].drop(columns=["_id","VITIMAS_SEM_INFO", "VITIMA_ILESA","VISIBILIDADE","CONDICAO_METERIOLOGICA","VEICULOS_ENVOLVIDOS"]).to_csv(f"data/{year}/p{year}.csv", index=False)

In [21]:
print(data.loc[(data.CARROÇA > 0)])

         _id                 DATA      HORA      CONCESSIONARIA     RODOVIA  \
11324  11325  2026-05-12T00:00:00  18:24:37  L23-LESTE PAULISTA  SPI179/060   
15320  15321  2026-05-17T00:00:00  20:34:44       L28-ENTREVIAS  SPA343/322   

          KM SENTIDO   LATITUDE  LONGITUDE                   CLASSE  ...  \
11324  5.000   NORTE -23.364704 -46.128761            ENGAVETAMENTO  ...   
15320  0.693   NORTE -21.092163 -48.048543  ATROPELAMENTO DE ANIMAL  ...   

      ÔNIBUS NÃO SE APLICA  MICRO-ÔNIBUS  TRATOR  UTILITÁRIO  CARRETINHA  \
11324    0.0           0.0           0.0     0.0         0.0         0.0   
15320    0.0           0.0           0.0     0.0         1.0         0.0   

       TRAILER  CARROÇA CICLOMOTOR CHARRETE  
11324      0.0      1.0        0.0      0.0  
15320      0.0      1.0        0.0      0.0  

[2 rows x 41 columns]
